# 04 — Join Strategies

**Concept:** Spark has four join strategies. The choice determines whether a join costs 0, 1, or 2 extra network shuffles — the single biggest driver of join performance.
**Core interview question:** *"A join that used to take 3 min now takes 20 min. Where do you start?"*

## What you will observe
1. `BroadcastHashJoin` (BHJ) — small table replicated to all executors, **zero shuffle** on the small side.
2. `SortMergeJoin` (SMJ) — both sides shuffled + sorted, then merged: **two shuffles**.
3. `ShuffleHashJoin` (SHJ) — **both sides shuffled** by join key; the smaller (build) side is loaded into a hash map in executor memory with no sort step.
4. `BroadcastNestedLoopJoin` (BNLJ) — nested loop for cross joins and non-equi (theta) joins.
5. How Spark selects a strategy and how to override it with hints.
6. The silent BHJ→SMJ regression: lookup table grows past `autoBroadcastJoinThreshold`.
7. Filter position: pushing filters before the join Exchange saves one full network round-trip.
8. Join skew and how AQE splits oversized shuffle partitions at runtime.


## How join nodes map between `explain()` and the Spark UI SQL tab

| `explain()` node | SQL tab icon | Shuffles | Stage cost |
|---|---|---|---|
| `BroadcastHashJoin` | BHJ diamond | 0 on probe side | 1–2 stages total |
| `BroadcastExchange` | Broadcast cylinder | — | small side only |
| `SortMergeJoin` | SMJ diamond | 2 (both sides) | 3+ stages |
| `ShuffleHashJoin` | SHJ diamond | 2 (both sides, no sort) | 2+ stages |
| `BroadcastNestedLoopJoin` | BNLJ | 0 or 1 | slow for large inputs |
| `Exchange hashpartitioning(key, N)` | shuffle arrow | 1 | stage boundary |
| `Sort [key ASC]` | sort arrow | 0 | required before SMJ merge |

**Reading the join plan bottom-to-top:**
- Each `Exchange` on a branch = shuffle for that side.
- BHJ: only one branch has a `BroadcastExchange` — the smaller side. The larger side has **no Exchange at all**.
- SMJ: **both branches** have `Exchange hashpartitioning` + `Sort` — two full shuffles.
- SHJ: **both branches** have `Exchange hashpartitioning` but **no `Sort`** nodes — two shuffles, build side hashed in memory (no sort step).


In [1]:
import sys
import os
from pathlib import Path

sys.path.append(str(Path().absolute().parent))

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("04-join-strategies")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.python.worker.faulthandler.enabled", "true")
    # Keep warehouse in a local temp dir so bucketed-table demos work
    .config("spark.sql.warehouse.dir", str(Path().absolute().parent / "data" / "warehouse"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version}  —  UI: http://localhost:4040")
print(f"autoBroadcastJoinThreshold : {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')}")
print(f"AQE enabled                : {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"shuffle.partitions         : {spark.conf.get('spark.sql.shuffle.partitions')}")


Spark 4.1.1  —  UI: http://localhost:4040
autoBroadcastJoinThreshold : 10485760b
AQE enabled                : true
shuffle.partitions         : 200


## Build sample datasets

Three tables with deliberately different sizes to trigger different join strategies:

| Table | Rows | Role |
|---|---|---|
| `countries` | 5 | Tiny lookup — will **always** auto-broadcast |
| `customers` | 150 | Medium — above broadcast threshold unless we lower it |
| `orders` | 5 000 | Fact table — the large probe side |

The `orders` data is **intentionally skewed**: customer_id = 1 owns 70 % of rows.
This becomes visible in the skew-join section.


In [2]:
import pandas as pd
import random
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)

random.seed(42)

# ── countries (5 rows — tiny, always broadcasts) ─────────────────────────────
countries_data = [
    (1, "US", "United States"),
    (2, "GB", "United Kingdom"),
    (3, "DE", "Germany"),
    (4, "FR", "France"),
    (5, "JP", "Japan"),
]
countries_schema = StructType([
    StructField("country_id",   IntegerType(), False),
    StructField("country_code", StringType(),  True),
    StructField("country_name", StringType(),  True),
])
countries = spark.createDataFrame(
    pd.DataFrame(countries_data, columns=["country_id","country_code","country_name"]),
    schema=countries_schema,
)

# ── customers (150 rows — medium, above default threshold with local data) ────
customers_data = [
    (i, f"customer_{i}", (i % 5) + 1)   # country_id cycles 1..5
    for i in range(1, 151)
]
customers_schema = StructType([
    StructField("customer_id",   IntegerType(), False),
    StructField("customer_name", StringType(),  True),
    StructField("country_id",    IntegerType(),  True),
])
customers = spark.createDataFrame(
    pd.DataFrame(customers_data, columns=["customer_id","customer_name","country_id"]),
    schema=customers_schema,
)

# ── orders (5 000 rows — skewed: customer_id=1 owns 70 % of rows) ────────────
orders_rows = []
for order_id in range(1, 5001):
    # Skew: 70 % of orders belong to customer 1
    if order_id <= 3500:
        cust_id = 1
    else:
        cust_id = random.randint(2, 150)
    orders_rows.append((
        order_id,
        cust_id,
        round(random.uniform(10.0, 2000.0), 2),
        (cust_id % 5) + 1,   # country_id aligned with customer
    ))

orders_schema = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", IntegerType(), True),
    StructField("amount",      DoubleType(),  True),
    StructField("country_id",  IntegerType(), True),
])
orders = spark.createDataFrame(
    pd.DataFrame(orders_rows, columns=["order_id","customer_id","amount","country_id"]),
    schema=orders_schema,
)

print(f"countries : {countries.count()} rows, {countries.rdd.getNumPartitions()} partitions")
print(f"customers : {customers.count()} rows, {customers.rdd.getNumPartitions()} partitions")
print(f"orders    : {orders.count()} rows,  {orders.rdd.getNumPartitions()} partitions")
print()
print("Order distribution by customer_id (top 5):")
orders.groupBy("customer_id").count().orderBy("count", ascending=False).show(5)


countries : 5 rows, 5 partitions
customers : 150 rows, 8 partitions
orders    : 5000 rows,  8 partitions

Order distribution by customer_id (top 5):
+-----------+-----+
|customer_id|count|
+-----------+-----+
|          1| 3500|
|        107|   22|
|         79|   22|
|        115|   18|
|        140|   16|
+-----------+-----+
only showing top 5 rows


## 1. BroadcastHashJoin (BHJ)

**When it fires:** One side of the join is estimated to be smaller than `spark.sql.autoBroadcastJoinThreshold` (default 10 MB).

**Mechanics:**
1. Spark serializes the small table into a hash map.
2. The driver broadcasts that hash map to every executor.
3. Each executor probes its local partition of the large table against the in-memory hash map.
4. **Zero shuffle on the large (probe) side.** Only the small side moves across the network — once.

**What to look for in the plan:**
```
BroadcastHashJoin [country_id], [country_id], Inner, BuildRight
:- Exchange hashpartitioning(...)        ← large side (orders)
+- BroadcastExchange HashedRelationBroadcastMode   ← small side (countries)
```
The `BroadcastExchange` is **only on the small side**. No `Exchange` on the probe (large) side.


In [3]:
# Reset threshold to default so auto-broadcast can fire
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

print("─" * 60)
print("JOIN: orders ⋈ countries (5 rows)  → expect BroadcastHashJoin")
print("─" * 60)

bhj = orders.join(countries, "country_id")

print("\nPhysical plan (read bottom-to-top):")
bhj.explain(mode="formatted")


────────────────────────────────────────────────────────────
JOIN: orders ⋈ countries (5 rows)  → expect BroadcastHashJoin
────────────────────────────────────────────────────────────

Physical plan (read bottom-to-top):
== Physical Plan ==
AdaptiveSparkPlan (6)
+- Project (5)
   +- BroadcastHashJoin Inner BuildRight (4)
      :- LocalTableScan (1)
      +- BroadcastExchange (3)
         +- LocalTableScan (2)


(1) LocalTableScan
Output [4]: [order_id#6, customer_id#7, amount#8, country_id#9]
Arguments: [order_id#6, customer_id#7, amount#8, country_id#9]

(2) LocalTableScan
Output [3]: [country_id#0, country_code#1, country_name#2]
Arguments: [country_id#0, country_code#1, country_name#2]

(3) BroadcastExchange
Input [3]: [country_id#0, country_code#1, country_name#2]
Arguments: HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=148]

(4) BroadcastHashJoin
Left keys [1]: [country_id#9]
Right keys [1]: [country_id#0]
Join type: Inner
Join condition: 

In [5]:
# Trigger execution so the Spark UI shows the completed job
t0 = __import__("time").perf_counter()
bhj_count = bhj.count()
print(f"Row count: {bhj_count}  ({__import__('time').perf_counter()-t0:.2f}s)")
print()
print("Post-execution plan (AQE isFinalPlan=true — check for BroadcastExchange):")
bhj.explain(mode="extended")

print()
print("Spark UI checklist for this job:")
print("  SQL tab : find 'BroadcastHashJoin' node")
print("  SQL tab : BroadcastExchange appears only on the countries branch (right/small side)")
print("  SQL tab : orders branch has NO Exchange node — zero shuffle on the large side")
print("  Stages  : 1–2 stages (scan orders + scan+broadcast countries → join)")


Row count: 5000  (0.30s)

Post-execution plan (AQE isFinalPlan=true — check for BroadcastExchange):
== Parsed Logical Plan ==
'Join UsingJoin(Inner, [country_id])
:- LocalRelation [order_id#6, customer_id#7, amount#8, country_id#9]
+- LocalRelation [country_id#0, country_code#1, country_name#2]

== Analyzed Logical Plan ==
country_id: int, order_id: int, customer_id: int, amount: double, country_code: string, country_name: string
Project [country_id#9, order_id#6, customer_id#7, amount#8, country_code#1, country_name#2]
+- Join Inner, (country_id#9 = country_id#0)
   :- LocalRelation [order_id#6, customer_id#7, amount#8, country_id#9]
   +- LocalRelation [country_id#0, country_code#1, country_name#2]

== Optimized Logical Plan ==
Project [country_id#9, order_id#6, customer_id#7, amount#8, country_code#1, country_name#2]
+- Join Inner, (country_id#9 = country_id#0)
   :- LocalRelation [order_id#6, customer_id#7, amount#8, country_id#9]
   +- LocalRelation [country_id#0, country_code#1, 

### BHJ Spark UI walkthrough

**SQL tab** (`/SQL/execution?id=N`):
- The plan has two branches feeding into `BroadcastHashJoin`.
- **Right branch (small side — countries):** `LocalTableScan` → `BroadcastExchange`. This is the serialized hash map broadcast to all executors.
- **Left branch (probe side — orders):** `LocalTableScan` only — **no Exchange node**. Orders never shuffle.
- The `BroadcastHashJoin` node label shows which side is `BuildRight` (the broadcast side).

**Stages tab:**
Expect 1–2 stages. There is no shuffle stage for `orders`. The broadcast happens as a side-channel from the driver — not reflected as a Spark shuffle stage.

**Task metrics:**
Each task reads its own partition of `orders` and probes the shared in-memory hash map. Task durations should be uniform — no skew possible from the join itself (though skew in orders partitions is still possible).

**Key rule:**
`BroadcastExchange` on one branch only = BHJ. If you see `Exchange hashpartitioning` on **both** branches, it is SMJ.


## 2. SortMergeJoin (SMJ)

**When it fires:** Both sides are too large to broadcast (above `autoBroadcastJoinThreshold`), **and** the join keys can be sorted.

**Mechanics:**
1. **Shuffle both sides** by the join key — rows with the same key land on the same executor.
2. **Sort both sides** within each partition — keys are now in ascending order.
3. **Merge pass** — two sorted iterators are walked in lock-step, producing matches.

This is **two full shuffles** — the most expensive strategy. But it is the only strategy that scales to arbitrarily large tables (it never builds a full in-memory hash map).

**What to look for in the plan:**
```
SortMergeJoin [customer_id], [customer_id], Inner
:- Sort [customer_id ASC]
:  +- Exchange hashpartitioning(customer_id, N)   ← shuffle 1 (orders)
+- Sort [customer_id ASC]
   +- Exchange hashpartitioning(customer_id, N)   ← shuffle 2 (customers)
```
Both branches have `Exchange` + `Sort`. That is the SMJ signature.


In [6]:
# Disable auto-broadcast to force Spark to use SortMergeJoin
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

print("─" * 60)
print("JOIN: orders ⋈ customers (150 rows), broadcast DISABLED")
print("Expected strategy: SortMergeJoin")
print("─" * 60)

smj = orders.join(customers, "customer_id")

print("\nPhysical plan:")
smj.explain(mode="formatted")


────────────────────────────────────────────────────────────
JOIN: orders ⋈ customers (150 rows), broadcast DISABLED
Expected strategy: SortMergeJoin
────────────────────────────────────────────────────────────

Physical plan:
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- SortMergeJoin Inner (7)
      :- Sort (3)
      :  +- Exchange (2)
      :     +- LocalTableScan (1)
      +- Sort (6)
         +- Exchange (5)
            +- LocalTableScan (4)


(1) LocalTableScan
Output [4]: [order_id#6, customer_id#7, amount#8, country_id#9]
Arguments: [order_id#6, customer_id#7, amount#8, country_id#9]

(2) Exchange
Input [4]: [order_id#6, customer_id#7, amount#8, country_id#9]
Arguments: hashpartitioning(customer_id#7, 200), ENSURE_REQUIREMENTS, [plan_id=350]

(3) Sort
Input [4]: [order_id#6, customer_id#7, amount#8, country_id#9]
Arguments: [customer_id#7 ASC NULLS FIRST], false, 0

(4) LocalTableScan
Output [3]: [customer_id#3, customer_name#4, country_id#5]
Arguments: [custom

In [7]:
t0 = __import__("time").perf_counter()
smj_count = smj.count()
print(f"Row count: {smj_count}  ({__import__('time').perf_counter()-t0:.2f}s)")
print()
print("Post-execution plan — count Exchange nodes (should be 2, one per side):")
smj.explain(mode="formatted")

# Restore threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

print()
print("Spark UI checklist:")
print("  SQL tab : find 'SortMergeJoin' node")
print("  SQL tab : BOTH branches have 'Exchange hashpartitioning' + 'Sort' — 2 shuffles")
print("  Stages  : 3+ stages (stage 0: shuffle orders, stage 1: shuffle customers,")
print("            stage 2: merge + output)  vs BHJ's 1–2 stages")


Row count: 5000  (0.81s)

Post-execution plan — count Exchange nodes (should be 2, one per side):
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- SortMergeJoin Inner (7)
      :- Sort (3)
      :  +- Exchange (2)
      :     +- LocalTableScan (1)
      +- Sort (6)
         +- Exchange (5)
            +- LocalTableScan (4)


(1) LocalTableScan
Output [4]: [order_id#6, customer_id#7, amount#8, country_id#9]
Arguments: [order_id#6, customer_id#7, amount#8, country_id#9]

(2) Exchange
Input [4]: [order_id#6, customer_id#7, amount#8, country_id#9]
Arguments: hashpartitioning(customer_id#7, 200), ENSURE_REQUIREMENTS, [plan_id=350]

(3) Sort
Input [4]: [order_id#6, customer_id#7, amount#8, country_id#9]
Arguments: [customer_id#7 ASC NULLS FIRST], false, 0

(4) LocalTableScan
Output [3]: [customer_id#3, customer_name#4, country_id#5]
Arguments: [customer_id#3, customer_name#4, country_id#5]

(5) Exchange
Input [3]: [customer_id#3, customer_name#4, country_id#5]
Arguments: hashpa

### SMJ Spark UI walkthrough

**SQL tab:**
- `SortMergeJoin` node at the top.
- **Left branch (orders):** `LocalTableScan` → `Exchange hashpartitioning(customer_id, N)` → `Sort [customer_id ASC]`.
- **Right branch (customers):** `LocalTableScan` → `Exchange hashpartitioning(customer_id, N)` → `Sort [customer_id ASC]`.
- Two `Exchange` nodes = two shuffles = two stage boundaries.

**Stages tab:**
3+ stages (with AQE, some may be coalesced on small data):
- Stage 0: scan + partial process on both sides → write shuffle files
- Stage 1: shuffle-read + sort on both sides
- Stage 2 (or merged with 1): the merge pass

**Compared to BHJ:**
SMJ on the same query as BHJ adds one extra `Exchange` (and its `Sort`) on the probe side — that is the cost of not broadcasting.
On large production data with 200+ executors, a single extra shuffle can add 2–5 minutes.


## 3. ShuffleHashJoin (SHJ)

**When it fires (rarely automatic):** Spark can choose SHJ over SMJ when:
- One side is significantly smaller (but still above broadcast threshold)
- `spark.sql.join.preferSortMergeJoin = false`
- The smaller side fits in executor memory after shuffling

**Mechanics:**
1. **Shuffle both sides** by the join key (same as SMJ).
2. **Build a hash map** from the smaller side (the "build" side) in executor memory.
3. **Probe** each row of the larger side against the hash map.

**Key difference from SMJ:** SHJ avoids the sort phase — cheaper per-task CPU, but the build side must fit in executor memory. SMJ streams both sides without building a full in-memory structure.

**Phases:**
1. Shuffle Phase: Spark redistributes both datasets across the cluster based on the join key. This ensures rows with the same key from both tables land on the exact same physical executor
2. Hash & Join Phase: On each executor, Spark takes the smaller dataset of the two, builds an in-memory Hash table for that partition, and then iterates through the larger dataset to find matches

**What to look for in the plan:**
```
ShuffleHashJoin [customer_id], [customer_id], Inner, BuildRight
:- Exchange hashpartitioning(customer_id, N)   ← probe side
+- Exchange hashpartitioning(customer_id, N)   ← build side (hashed in memory)
```
Both branches have `Exchange` but **no `Sort`** nodes — that is the SHJ signature.


In [8]:
# SHJ is rarely selected automatically in modern Spark.
# Use the shuffle_hash hint to force it.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

print("─" * 60)
print("JOIN: orders ⋈ customers with shuffle_hash hint")
print("Expected strategy: ShuffleHashJoin")
print("─" * 60)

shj = orders.hint("shuffle_hash").join(customers, "customer_id")
shj.explain(mode="formatted")

# Compare with SMJ side-by-side
print()
print("─" * 60)
print("SMJ plan for same join (no hint, broadcast disabled):")
print("─" * 60)
smj_compare = orders.join(customers, "customer_id")
smj_compare.explain(mode="formatted")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))


────────────────────────────────────────────────────────────
JOIN: orders ⋈ customers with shuffle_hash hint
Expected strategy: ShuffleHashJoin
────────────────────────────────────────────────────────────
== Physical Plan ==
AdaptiveSparkPlan (7)
+- Project (6)
   +- ShuffledHashJoin Inner BuildLeft (5)
      :- Exchange (2)
      :  +- LocalTableScan (1)
      +- Exchange (4)
         +- LocalTableScan (3)


(1) LocalTableScan
Output [4]: [order_id#6, customer_id#7, amount#8, country_id#9]
Arguments: [order_id#6, customer_id#7, amount#8, country_id#9]

(2) Exchange
Input [4]: [order_id#6, customer_id#7, amount#8, country_id#9]
Arguments: hashpartitioning(customer_id#7, 200), ENSURE_REQUIREMENTS, [plan_id=537]

(3) LocalTableScan
Output [3]: [customer_id#3, customer_name#4, country_id#5]
Arguments: [customer_id#3, customer_name#4, country_id#5]

(4) Exchange
Input [3]: [customer_id#3, customer_name#4, country_id#5]
Arguments: hashpartitioning(customer_id#3, 200), ENSURE_REQUIREMENTS, [

In [ ]:
t0 = __import__("time").perf_counter()
shj_count = shj.count()
print(f"Row count: {shj_count}  ({__import__('time').perf_counter()-t0:.2f}s)")
print()
print("Post-execution plan — count Exchange nodes (should be 2, one per side):")
shj.explain(mode="formatted")

# Restore threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

print()
print("Spark UI checklist:")
print("  SQL tab : find 'ShuffleHashJoin' node")
print("  SQL tab : BOTH branches have 'Exchange hashpartitioning' but no 'Sort' — 2 shuffles but no sort")
print("  Stages  : 3+ stages (stage 0: shuffle orders, stage 1: shuffle customers,")
print("            stage 2: merge + output)  vs BHJ's 1–2 stages")

Row count: 5000  (0.75s)

Post-execution plan — count Exchange nodes (should be 2, one per side):
== Physical Plan ==
AdaptiveSparkPlan (7)
+- Project (6)
   +- ShuffledHashJoin Inner BuildLeft (5)
      :- Exchange (2)
      :  +- LocalTableScan (1)
      +- Exchange (4)
         +- LocalTableScan (3)


(1) LocalTableScan
Output [4]: [order_id#6, customer_id#7, amount#8, country_id#9]
Arguments: [order_id#6, customer_id#7, amount#8, country_id#9]

(2) Exchange
Input [4]: [order_id#6, customer_id#7, amount#8, country_id#9]
Arguments: hashpartitioning(customer_id#7, 200), ENSURE_REQUIREMENTS, [plan_id=537]

(3) LocalTableScan
Output [3]: [customer_id#3, customer_name#4, country_id#5]
Arguments: [customer_id#3, customer_name#4, country_id#5]

(4) Exchange
Input [3]: [customer_id#3, customer_name#4, country_id#5]
Arguments: hashpartitioning(customer_id#3, 200), ENSURE_REQUIREMENTS, [plan_id=538]

(5) ShuffledHashJoin
Left keys [1]: [customer_id#7]
Right keys [1]: [customer_id#3]
Join type

### SHJ vs SMJ: what changes in the plan

| Feature | SortMergeJoin | ShuffleHashJoin |
|---|---|---|
| Both sides shuffled? | Yes | Yes |
| Sort required? | Yes (both sides) | **No** |
| Memory requirement | Streaming — low | Build side fits in RAM |
| Safe for large tables? | Yes (spills to disk) | Risk of OOM on build side |
| When to prefer | Default for large joins | One side fits in memory, CPU is the bottleneck |

**Interview note:** SHJ avoids sorting but requires the build side to fit in executor execution memory. **If it doesn't, Spark will OOM or fall back to SMJ. Prefer SMJ for production joins on unknown-size tables**.


## 4. BroadcastNestedLoopJoin (BNLJ)

**When it fires:**
1. **Non-equi (theta) join** — the join condition is not a simple key equality (e.g., `orders.amount BETWEEN band.low AND band.high`).
2. **Cross join** — no join condition (cartesian product).
3. **Unsupported join types** for other strategies (e.g., full outer with no equi-condition).

**Mechanics:**
For each row of the probe side, scan **every row** of the build side. This is O(n × m) — the slowest possible join.

**Why it matters:**
A cross join between a 10 M-row fact table and a 1 000-row lookup table produces **10 billion rows**.
Always confirm: is the BNLJ intentional, or is a missing join key creating an accidental cartesian product?

**What to look for in the plan:**
```
BroadcastNestedLoopJoin BuildRight, Cross
+- BroadcastExchange IdentityBroadcastMode
```


In [10]:
# ── Example 1: Cross join (cartesian product) ────────────────────────────────
print("─" * 60)
print("CROSS JOIN: countries × countries  → BroadcastNestedLoopJoin")
print("─" * 60)

# Must enable crossJoin explicitly (safety guard)
spark.conf.set("spark.sql.crossJoin.enabled", "true")

cross = countries.crossJoin(countries.toDF(*[c + "_r" for c in countries.columns]))
cross.explain(mode="formatted")
cross_count = cross.count()
print(f"5 × 5 = {cross_count} rows (cartesian product)")

print()
print("─" * 60)
print("NON-EQUI (theta) JOIN: orders where amount falls in a band")
print("─" * 60)

# Small range-band table
import pandas as pd
bands_pd = pd.DataFrame([
    (1, 0.0,    500.0,  "low"),
    (2, 500.0,  1000.0, "medium"),
    (3, 1000.0, 9999.0, "high"),
], columns=["band_id", "min_amount", "max_amount", "tier"])

from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType
bands = spark.createDataFrame(bands_pd, schema=StructType([
    StructField("band_id",    IntegerType(), False),
    StructField("min_amount", DoubleType(),  True),
    StructField("max_amount", DoubleType(),  True),
    StructField("tier",       StringType(),  True),
]))

# Non-equi condition: amount BETWEEN min_amount AND max_amount
theta = orders.join(
    bands,
    (orders["amount"] >= bands["min_amount"]) & (orders["amount"] < bands["max_amount"])
)
theta.explain(mode="formatted")
theta_count = theta.count()
print(f"Theta join produced {theta_count} rows")

spark.conf.set("spark.sql.crossJoin.enabled", "false")


────────────────────────────────────────────────────────────
CROSS JOIN: countries × countries  → BroadcastNestedLoopJoin
────────────────────────────────────────────────────────────
== Physical Plan ==
AdaptiveSparkPlan (5)
+- BroadcastNestedLoopJoin Cross BuildRight (4)
   :- LocalTableScan (1)
   +- BroadcastExchange (3)
      +- LocalTableScan (2)


(1) LocalTableScan
Output [3]: [country_id#0, country_code#1, country_name#2]
Arguments: [country_id#0, country_code#1, country_name#2]

(2) LocalTableScan
Output [3]: [country_id_r#87, country_code_r#88, country_name_r#89]
Arguments: [country_id_r#87, country_code_r#88, country_name_r#89]

(3) BroadcastExchange
Input [3]: [country_id_r#87, country_code_r#88, country_name_r#89]
Arguments: IdentityBroadcastMode, [plan_id=700]

(4) BroadcastNestedLoopJoin
Join type: Cross
Join condition: None

(5) AdaptiveSparkPlan
Output [6]: [country_id#0, country_code#1, country_name#2, country_id_r#87, country_code_r#88, country_name_r#89]
Arguments: 

### BNLJ: the accidental cartesian blast radius

In production, BNLJ most often appears due to a **bug** — a join key is missing or misnamed:

```python
# Bug: joining on wrong column names — produces cartesian product
df_a.join(df_b, df_a["order_id"] == df_b["order_num"])  # OK if keys match
df_a.join(df_b)                                           # CROSS JOIN — BNLJ!
```

**Detection signals:**
- `BroadcastNestedLoopJoin` in `explain()` output where you expected BHJ or SMJ.
- Stage duration climbing exponentially with data growth.
- Output row count is much larger than either input table.

**Interview answer to "how do you spot an accidental cartesian":**
Check `explain()` for `BroadcastNestedLoopJoin` or a `Cross` join type. Verify output row count — it should be ≤ max(left, right) rows for a normal join. If it is left × right, a join condition is missing.


## 5. Explicit join hints

Spark provides four hints that **override Catalyst's automatic strategy selection**. Use them when:
- Catalyst makes the wrong choice because statistics are stale or absent.
- You are debugging a plan and want to force a specific strategy to compare.
- AQE cannot help because the table size estimate is wrong at plan time.

| Hint | Strategy forced | Usage |
|---|---|---|
| `broadcast(df)` or `df.hint("broadcast")` | BroadcastHashJoin | Small table you *know* fits in memory |
| `df.hint("merge")` | SortMergeJoin | Force SMJ even when BHJ would fire |
| `df.hint("shuffle_hash")` | ShuffleHashJoin | Avoid sort cost when build side fits in RAM |
| `df.hint("shuffle_replicate_nl")` | BroadcastNestedLoopJoin | Rarely used; testing only |

**Hint placement:** the hint is applied to the DataFrame it is called on — that side becomes the **build side** (broadcast side for BHJ, hash side for SHJ).


In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # disable auto-broadcast

print("=" * 60)
print("HINT: broadcast()  →  BroadcastHashJoin")
print("=" * 60)
hint_bhj = orders.join(broadcast(customers), "customer_id")
hint_bhj.explain()

print()
print("=" * 60)
print("HINT: merge  →  SortMergeJoin")
print("=" * 60)
hint_smj = orders.join(customers.hint("merge"), "customer_id")
hint_smj.explain()

print()
print("=" * 60)
print("HINT: shuffle_hash  →  ShuffleHashJoin")
print("=" * 60)
hint_shj = orders.hint("shuffle_hash").join(customers, "customer_id")
hint_shj.explain()

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))


### When to use hints vs. when to trust Catalyst

**Trust Catalyst when:** statistics are available and up to date (tables analyzed via `ANALYZE TABLE`, or reading from Parquet with statistics in the file footer).

**Use `broadcast()` when:**
- A lookup table grows occasionally and you *know* it will stay under a size you can broadcast.
- You have verified in `explain()` that Catalyst is choosing SMJ even though the table is small — usually because statistics are missing.

**Never use `merge()` / `shuffle_hash()` in production code unless you have a documented reason.** Hints bypass Catalyst and will silently produce the wrong plan if the data characteristics change.

**AQE caveat:** even with a wrong static plan, AQE can convert SMJ→BHJ at runtime (see notebook 06). Hints prevent this — if you force SMJ with `.hint("merge")`, AQE cannot override it.


## 6. The silent BHJ → SMJ regression

**The pattern:**
1. At pipeline launch, a lookup table is 8 MB → Catalyst chooses BHJ.
2. Six months later the lookup table is 12 MB → Catalyst silently switches to SMJ.
3. Runtime triples. No code changed. No alert fired.

**Why it is silent:**
- There is no warning log entry when Catalyst chooses SMJ over BHJ.
- The plan change is only visible in `explain()` output or the Spark UI SQL tab.
- Metrics show the job is "slower" but the root cause is the join strategy change.

**Detection:**
1. Open the Spark UI SQL tab for the slow job.
2. Compare current plan vs a saved baseline — look for `BroadcastHashJoin` → `SortMergeJoin`.
3. Count `Exchange` nodes: BHJ typically has 0–1; SMJ has 2.

**The interview question form:** *"You added a new dataset to a pipeline and the runtime tripled. How do you investigate whether a join strategy changed?"*


In [11]:
# ── Simulate the regression by manipulating the broadcast threshold ───────────

# STATE 1: Threshold = 10 MB → customers (small data) broadcasts → BHJ
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

print("STATE 1: autoBroadcastJoinThreshold = 10 MB")
plan_before = orders.join(customers, "customer_id")
plan_before.explain()

print()
print("─" * 60)
print()

# STATE 2: Threshold lowered to 1 byte → nothing broadcasts → SMJ
# This simulates what happens when the lookup table grows past the threshold.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "1")

print("STATE 2: autoBroadcastJoinThreshold = 1 byte (simulates lookup table growth)")
plan_after = orders.join(customers, "customer_id")
plan_after.explain()

print()
print("OBSERVATION:")
print("  Before: BroadcastHashJoin — 0 shuffles on orders side")
print("  After : SortMergeJoin     — 2 shuffles (one per side)")
print("  The only change was the threshold. Same query, same data, different strategy.")

# Restore
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))


STATE 1: autoBroadcastJoinThreshold = 10 MB
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#7, order_id#6, amount#8, country_id#9, customer_name#4, country_id#5]
   +- BroadcastHashJoin [customer_id#7], [customer_id#3], Inner, BuildRight, false
      :- LocalTableScan [order_id#6, customer_id#7, amount#8, country_id#9]
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=895]
         +- LocalTableScan [customer_id#3, customer_name#4, country_id#5]



────────────────────────────────────────────────────────────

STATE 2: autoBroadcastJoinThreshold = 1 byte (simulates lookup table growth)
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#7, order_id#6, amount#8, country_id#9, customer_name#4, country_id#5]
   +- SortMergeJoin [customer_id#7], [customer_id#3], Inner
      :- Sort [customer_id#7 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(custome

### Fixing the BHJ→SMJ regression

**Option 1 — Explicit broadcast hint:**
```python
# Pin the strategy: even if the table grows past the threshold, BHJ fires
result = orders.join(broadcast(lookup_table), "key")
```
Risk: if the table grows large enough that broadcasting OOMs executors, the job fails rather than gracefully degrading to SMJ.

**Option 2 — Raise `autoBroadcastJoinThreshold`:**
```python
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(50 * 1024 * 1024))  # 50 MB
```
Works if the table is genuinely small. Bad if the table can grow unboundedly.

**Option 3 — AQE runtime conversion (notebook 06):**
With `spark.sql.adaptive.enabled=true`, AQE can convert SMJ→BHJ at runtime after Stage 0 writes shuffle files, if it sees the build side is actually small. This is the safest option — it handles both cases automatically.

**Monitoring recommendation:** record the join strategy in a job metadata table on every run. Alert if it changes from BHJ to SMJ.


## 7. Filter position relative to the join

**Rule:** filters that can be evaluated before the join should appear **below** the `Exchange` node in the physical plan — they reduce the data that participates in the shuffle.

A filter that appears **above** the join `Exchange` shuffles the full dataset first and then discards rows — wasted network I/O.

**Catalyst usually handles this automatically** (predicate pushdown). But there are cases where pushdown does not fire:
- Filters on derived columns computed inside the join.
- Filters referencing both sides of the join.
- Certain UDFs that Catalyst cannot reason about.

**What to check in the plan:**
```
Filter (amount > 500)                   ← LATE filter: after Exchange
+- SortMergeJoin [customer_id]
   :- Exchange hashpartitioning(...)    ← full dataset shuffled
```
vs
```
SortMergeJoin [customer_id]             ← EARLY filter: before Exchange
:- Exchange hashpartitioning(...)
   +- Filter (amount > 500)             ← rows discarded before shuffle
```


In [13]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# ── Plan A: filter AFTER join ─────────────────────────────────────────────────
late_filter = (
    orders
    .join(customers, "customer_id")    # shuffle all 5000 orders
    .filter(F.col("amount") > 1500)    # then discard 90 % of rows
)
print("PLAN A — filter after join:")
late_filter.explain(mode="extended")

print()

# ── Plan B: filter BEFORE join ────────────────────────────────────────────────
# Catalyst's predicate pushdown moves the filter below the Exchange automatically
early_filter = (
    orders
    .filter(F.col("amount") > 1500)    # discard rows first
    .join(customers, "customer_id")    # then shuffle the smaller dataset
)
print("PLAN B — filter before join:")
early_filter.explain(mode="extended")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))


PLAN A — filter after join:
== Parsed Logical Plan ==
'Filter '`>`('amount, 1500)
+- Project [customer_id#7, order_id#6, amount#8, country_id#9, customer_name#4, country_id#5]
   +- Join Inner, (customer_id#7 = customer_id#3)
      :- LocalRelation [order_id#6, customer_id#7, amount#8, country_id#9]
      +- LocalRelation [customer_id#3, customer_name#4, country_id#5]

== Analyzed Logical Plan ==
customer_id: int, order_id: int, amount: double, country_id: int, customer_name: string, country_id: int
Filter (amount#8 > cast(1500 as double))
+- Project [customer_id#7, order_id#6, amount#8, country_id#9, customer_name#4, country_id#5]
   +- Join Inner, (customer_id#7 = customer_id#3)
      :- LocalRelation [order_id#6, customer_id#7, amount#8, country_id#9]
      +- LocalRelation [customer_id#3, customer_name#4, country_id#5]

== Optimized Logical Plan ==
Project [customer_id#7, order_id#6, amount#8, country_id#9, customer_name#4, country_id#5]
+- Join Inner, (customer_id#7 = customer_id#

### What to observe in the two plans

**Plan A (filter after join):** Look for the `Filter` node **above** the `SortMergeJoin`. Both Exchange nodes shuffle the full `orders` dataset (5 000 rows) before the filter discards most of them.

**Plan B (filter before join):** Catalyst's predicate pushdown moves the `Filter` node **below** the `Exchange` on the `orders` branch. Only the ~300 rows passing `amount > 1500` enter the shuffle — 94 % reduction in shuffle data.

**Catalyst usually does this automatically.** The lesson is to understand *when* it fires and how to verify it did:
- Open `explain(mode="formatted")`.
- Find the `Filter` node and check its position relative to `Exchange`.
- If the filter is above an Exchange that it could logically be below → pushdown did not fire → manual rewrite or `spark.sql.optimizer.nestedSchemaPruning.enabled`.

**Interview answer:** *"I would check `explain()` to see if the filter appears below the Exchange (pushed down) or above it (late filter). If it is above, I would rewrite the query to filter before the join, or investigate why predicate pushdown did not fire."*


## 8. Join skew and AQE skew-join optimization

**The problem:**
After the shuffle, the partition for `customer_id = 1` holds 70 % of all `orders` rows. One task processes 3 500 rows while 199 other tasks process ~7 rows each. The stage duration equals the slowest task.

**Static options (pre-AQE):**
1. **Salting** — append a random integer to the join key on the skewed side, explode the dimension side N times, join on the salted key. Extra shuffle + more complex code.
2. **Repartitioning** — not effective for key-based skew.

**AQE skew-join optimization (Spark 3.0+):**
After Stage 0 (the shuffle) completes, AQE reads the actual shuffle file sizes. If one partition is `skewedPartitionFactor` × median and larger than `skewedPartitionThresholdInBytes`, AQE:
1. Splits the oversized partition into sub-partitions.
2. **Duplicates** the corresponding rows from the other side to match.
3. Processes the sub-partitions in parallel.

This happens **transparently at runtime** — no code change needed. The post-execution plan shows `AQEShuffleRead` with a split partition annotation.


In [ ]:
# Show the skew in the data first
print("Order count by customer_id (top 5 to confirm skew):")
orders.groupBy("customer_id").count().orderBy("count", ascending=False).show(5)

print()
print("─" * 60)
print("Skewed join: orders ⋈ customers on customer_id")
print("customer_id=1 holds 70 % of orders → one partition gets 3500 rows")
print("─" * 60)

# Disable auto-broadcast so we actually get a shuffle join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# Make AQE skew sensitivity very aggressive so it fires on our small dataset
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "2")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "1b")

skewed_join = orders.join(customers, "customer_id")

print("Pre-execution plan (isFinalPlan=false — AQE not yet applied):")
skewed_join.explain(mode="formatted")


In [ ]:
import time

t0 = time.perf_counter()
skewed_count = skewed_join.count()
elapsed = time.perf_counter() - t0
print(f"Row count: {skewed_count}  ({elapsed:.2f}s)")

print()
print("Post-execution plan (isFinalPlan=true — AQE decisions visible):")
skewed_join.explain(mode="formatted")

print()
print("Spark UI checklist for this job:")
print("  SQL tab   : find 'CustomShuffleReader' or 'AQEShuffleRead' on the skewed branch")
print("  SQL tab   : plan shows 'skewed' annotation on the split partition")
print("  Stages tab: may show extra sub-stages for the split partitions")
print("  Task Metrics: customer_id=1 partition appears as multiple smaller tasks")

# Restore
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256mb")


### AQE skew join vs manual salting

| | AQE skew join | Manual salting |
|---|---|---|
| **When it works** | Both sides in the same Spark job | Pre-aggregation before the join; cross-job pipelines |
| **Code change required?** | None | Yes — add salt column, explode dimension side |
| **Visibility** | Plan shows `AQEShuffleRead` with split marker | Plan shows extra shuffle stage for salting |
| **Extra shuffle?** | No — uses the existing shuffle files | Yes — one additional shuffle to join on salted key |
| **Limitations** | Requires `spark.sql.adaptive.enabled=true`; splits are bounded by `spark.sql.adaptive.skewJoin.maxSkewedPartitionFactor` | Can produce data quality bugs if dimension side is not correctly exploded |

**Interview answer:** *"I would first enable AQE's skew join optimizer — it handles the most common case with no code change. If AQE is not available, or if the skew is caused by a pre-aggregation step rather than the join itself, I would use salting with a two-pass aggregation."*


## 9. Bucketing: pre-shuffled tables

**What it is:**
Bucketing writes a table's rows into a fixed number of files ("buckets"), partitioned by the hash of a designated key column — exactly the same hashing that a shuffle would apply. When two tables are bucketed on the same key with the same bucket count, **the join shuffle is already done** at write time.

**When to use it:**
- A dimension table is joined repeatedly in many downstream jobs.
- The join key does not change between jobs.
- The table is large enough that saving one shuffle per join across N jobs justifies the bucketed write cost.

**What to look for in the plan:**
- Normal SMJ plan: `Exchange hashpartitioning(key, N)` on both branches.
- Bucketed join plan: **no `Exchange` nodes** — the plan shows `FileScan` directly into `SortMergeJoin` (or even skips the sort if `sortBy` was specified).

**Limitations:**
- Bucketing only works for tables stored in Spark-managed warehouses (Hive-compatible).
- Bucket count must match on both tables for the optimization to fire.
- On GCS/S3 with many small bucket files, the small file problem can offset the shuffle savings.


In [ ]:
import tempfile, shutil, os

# Use a local warehouse directory (required for managed tables / bucketing)
warehouse_path = str(Path().absolute().parent / "data" / "warehouse")
os.makedirs(warehouse_path, exist_ok=True)

print("─" * 60)
print("Writing bucketed tables (simulates pre-shuffled storage)")
print("─" * 60)

# Write orders bucketed by customer_id into 8 buckets
(
    orders
    .write
    .mode("overwrite")
    .bucketBy(8, "customer_id")
    .sortBy("customer_id")
    .saveAsTable("orders_bucketed")
)
print("orders_bucketed written (8 buckets, sorted by customer_id)")

# Write customers bucketed by customer_id into 8 buckets (MUST match)
(
    customers
    .write
    .mode("overwrite")
    .bucketBy(8, "customer_id")
    .sortBy("customer_id")
    .saveAsTable("customers_bucketed")
)
print("customers_bucketed written (8 buckets, sorted by customer_id)")

# Read back and join — Spark should detect the bucketing and skip the shuffle
orders_b    = spark.table("orders_bucketed")
customers_b = spark.table("customers_bucketed")

print()
print("─" * 60)
print("Plan for bucketed join (expect NO Exchange nodes):")
print("─" * 60)
bucketed_join = orders_b.join(customers_b, "customer_id")
bucketed_join.explain(mode="formatted")

print()
print("─" * 60)
print("Plan for non-bucketed join (same data, broadcast disabled):")
print("─" * 60)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
normal_join = orders.join(customers, "customer_id")
normal_join.explain(mode="formatted")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))


### What to observe in the bucketed vs normal plan

**Bucketed plan:** the `FileScan` nodes read directly from the pre-sorted, pre-partitioned bucket files. There are **no `Exchange` nodes** — Spark knows the data is already co-partitioned. The `SortMergeJoin` (or `BucketedScan`) proceeds directly.

**Normal plan:** `FileScan` → `Exchange hashpartitioning(customer_id, N)` on both branches → `SortMergeJoin`. Two full shuffles.

**Stage count:**
- Bucketed: 1 stage (read both sides in parallel → merge).
- Normal: 3+ stages (read + shuffle orders, read + shuffle customers, merge).

**Production pattern:**
Write the shared dimension table once per day (or whenever it changes) in bucketed format. All downstream jobs that join on the same key skip the shuffle automatically.

**Note:** on local mode with small data, you may still see sort nodes (`Sort`) even with bucketing, because Spark verifies that sort order is preserved. On a real cluster reading from HDFS/GCS with proper statistics, the sort is also skipped.


## Spark UI checklist for join analysis

Run this checklist for every join query in a slow pipeline.

### Step 1 — SQL tab: identify the join strategy

1. Open the SQL tab for the slow job.
2. Find the join node — it will be labelled `BroadcastHashJoin`, `SortMergeJoin`, `ShuffleHashJoin`, or `BroadcastNestedLoopJoin`.
3. Count `Exchange` nodes on **each branch** feeding into the join:

| Exchange count | Strategy | Shuffle cost |
|---|---|---|
| One `BroadcastExchange` (small side); zero `Exchange hashpartitioning` | BHJ | 0 data shuffles — small side broadcast only |
| Two `Exchange hashpartitioning`, **no `Sort`** | SHJ | 2 shuffles, build side hashed in memory |
| Two `Exchange hashpartitioning` + `Sort` on **both** sides | SMJ | 2 shuffles + sort |
| `Cross` annotation | BNLJ | O(n×m) nested loop — investigate! |
| `Cross` annotation | BNLJ | O(n × m) — investigate! |

### Step 2 — SQL tab: verify filter position

For each `Filter` node, check whether it appears **below** (before) or **above** (after) the nearest `Exchange` node on the same branch:
- Below = pushed down ✓ — data is filtered before it enters the shuffle.
- Above = late filter ✗ — full dataset was shuffled, then rows were discarded.

### Step 3 — Stages tab: count stages

- BHJ: 1–2 stages.
- SMJ: 3+ stages (two shuffle stages + merge).
- Unexpected stage increase between baseline and current run = strategy likely changed.

### Step 4 — Task metrics: detect skew

Click into the shuffle-read stage (the stage after the Exchange):
- **Duration histogram:** one bar far to the right = one task processes a disproportionate partition (hot key skew).
- **Input Records per task:** if one task shows 10× the records of others = confirmed skew.
- If `spark.sql.adaptive.skewJoin.enabled=true`, post-execution plan should show `AQEShuffleRead` with split markers on skewed partitions.

### Debugging decision tree for slow joins

```
Join is slow →
  SQL tab: what is the join strategy?
    BroadcastNestedLoopJoin?
      YES → accidental cross join / missing condition — fix the query
    SortMergeJoin where BHJ is expected?
      YES → lookup table grew past autoBroadcastJoinThreshold
            Fix: F.broadcast() hint, raise threshold, or enable AQE (SMJ→BHJ at runtime)
    SortMergeJoin (expected) but stage is slow?
      Stages tab: which stage is the bottleneck — the shuffle write or shuffle read?
      Task Metrics: Duration histogram skewed?
        ONE OUTLIER → hot key skew on join column
                      Fix: AQE skew join (enabled by default) or manual salting
        ALL SLOW    → data volume, not skew — tune parallelism or add resources
      Filter above Exchange?
        YES → late filter — push upstream before the join
```


## Interview questions — answer from memory before moving on

### Q1. What are the four join strategies in Spark? When does each apply?

**A:**
- **BroadcastHashJoin:** one side is below `autoBroadcastJoinThreshold` (default 10 MB). Small side is serialized into a hash map and broadcast to all executors. Zero shuffles on the probe side. Best strategy when available.
- **SortMergeJoin:** both sides are above the broadcast threshold and the join is an equi-join. Both sides shuffle by join key, sort, then merge. Two full shuffles — most expensive but scales to any table size. The default for large-large joins.
- **ShuffleHashJoin:** both side shuffles, the smaller side builds a hash map in executor memory. No sort step. Spark rarely chooses this automatically; use `shuffle_hash` hint to force it. Risk: build side must fit in execution memory.
- **BroadcastNestedLoopJoin:** non-equi (theta) joins and cross joins. O(n × m) nested loop. Usually appears as a bug (missing join condition) — always verify output row count.

---

### Q2. How does Spark decide whether to use a broadcast join? What are the risks of broadcasting a large table?

**A:**
Spark estimates the size of each join side using table statistics (from `ANALYZE TABLE` or Parquet file statistics). If the estimated size is below `spark.sql.autoBroadcastJoinThreshold` (default 10 MB), Catalyst plans a BHJ with that side as the build side. AQE can also convert SMJ→BHJ at runtime after seeing actual shuffle file sizes.

**Risks of broadcasting a large table:**
1. **Driver OOM:** the driver serializes the table and holds the broadcast variable in heap. If the table is 2 GB and the driver has 4 GB heap, it may OOM.
2. **Executor OOM:** each executor holds its own copy of the broadcast variable in memory. With 100 executors, a 1 GB broadcast consumes 100 GB of aggregate executor memory.
3. **Long serialization time:** a large broadcast adds latency at the start of every job that uses it, even for jobs that process only a few rows.

---

### Q3. You added a new dataset to a pipeline and the runtime tripled. How do you investigate whether a join strategy changed?

**A:**
1. Open the Spark UI SQL tab for the current run. Find the join node — note the strategy (`BroadcastHashJoin` vs `SortMergeJoin`).
2. Compare against a saved baseline plan (or a run from before the dataset addition).
3. Count `Exchange` nodes on each branch: BHJ = 0 on probe side; SMJ = 2.
4. If the strategy changed from BHJ to SMJ, the most likely cause is a table exceeding `autoBroadcastJoinThreshold`. Check the table size.
5. Fix: add an explicit `broadcast()` hint, raise the threshold, or wait for AQE to convert SMJ→BHJ at runtime (if AQE is enabled and the table is genuinely small).

---

### Q4. What is bucketing in Spark? What join scenario does it optimize?

**A:**
Bucketing writes a table's data into N files, where each file contains rows whose join key hashes to that bucket number — the same partitioning that a shuffle would apply. When two tables are bucketed by the same key with the same N, a join on that key skips the shuffle entirely: both sides read their pre-partitioned files directly into the join operator.

Best suited for: a shared dimension table joined repeatedly across many downstream jobs. The bucketed write cost is amortized across all jobs that reuse it.

Limitations: both tables must use the same bucket count; only works for Spark-managed warehouse tables; bucket file management (small files) requires monitoring.

---

### Q5. A join between a 10B-row fact table and a 1M-row dimension table is slow. What strategies would you consider?

**A (in order):**
1. **BHJ:** estimate the serialized size of the 1M-row dimension table. If it is under 10–50 MB (common for narrow tables), raise `autoBroadcastJoinThreshold` or add a `broadcast()` hint. Zero shuffles on the fact side.
2. **AQE SMJ→BHJ conversion:** if static stats are wrong, enable AQE — it will broadcast the dimension side after Stage 0 if the actual shuffle size is below the threshold.
3. **Bucketing:** if this join runs daily and the dimension key is stable, bucket both tables on the join key. Eliminates the shuffle for all future runs.
4. **Filter-before-join:** if the query filters a subset of the fact table, verify the filter is pushed below the Exchange (predicate pushdown). Reduces the volume that enters the shuffle.
5. **AQE skew join:** if the dimension key has hot keys that create large fact-table partitions after the shuffle, enable AQE skew join to split them.

---

### Q6. How does AQE handle skewed join keys differently than static Catalyst?

**A:**
Static Catalyst plans the join before any data moves. It has no way to know that `customer_id = 1` owns 70 % of rows — it uses uniform statistics. The shuffle runs, one partition receives 3 500 rows, and one task takes 10× as long as all others.

AQE intervenes **after** the shuffle stage completes. It reads the actual shuffle file sizes. If one file is `skewedPartitionFactor` × the median file size AND larger than `skewedPartitionThresholdInBytes`, AQE:
1. Splits that oversized shuffle file into sub-partitions.
2. Duplicates the matching rows from the other side for each sub-partition.
3. Processes the sub-partitions as separate tasks in parallel.

The result is that the hot-key partition is handled by multiple tasks instead of one. No code change required. The post-execution plan shows `AQEShuffleRead` with a "skewed" annotation.

The key difference from manual salting: AQE acts at runtime with actual data statistics; salting requires upfront code changes and adds an extra shuffle stage.


## Key takeaways — cheat sheet

| Strategy | Shuffles | Memory requirement | When Spark chooses it |
|---|---|---|---|
| BroadcastHashJoin | 0 on probe side | Build side fits in executor heap | Small side < `autoBroadcastJoinThreshold` |
| SortMergeJoin | 2 (both sides) | Streaming — low | Large-large equi-join (default) |
| ShuffleHashJoin | 2 (both sides). No sort | Build side fits in executor memory (build hash table on smaller) | Explicit hint or `preferSortMergeJoin=false` |
| BroadcastNestedLoopJoin | 0 or 1 | Broadcast side in memory | Non-equi or cross join |

**The one rule:** count `Exchange` nodes in `explain()`. Each one is a shuffle, a stage boundary, and a network round-trip for all your data. Minimize them.

**Debugging order for slow joins:**
1. SQL tab → identify strategy.
2. Strategy changed from BHJ to SMJ? → table grew past broadcast threshold.
3. SMJ and task duration histogram shows one outlier? → join skew → enable AQE skew join.
4. Filter above Exchange? → late filter → push upstream.
5. Join runs repeatedly on the same key? → consider bucketing.

**Next notebook:** `05_catalyst_optimizer.ipynb` — how Catalyst selects these strategies through rule-based and cost-based optimization, and what `ANALYZE TABLE` actually changes.


In [ ]:
# Drop managed tables created by the bucketing demo
try:
    spark.sql("DROP TABLE IF EXISTS orders_bucketed")
    spark.sql("DROP TABLE IF EXISTS customers_bucketed")
    print("Managed tables dropped.")
except Exception as e:
    print(f"Cleanup warning (non-fatal): {e}")

print("Notebook complete. Spark UI still available at http://localhost:4040")
print("Run spark.stop() to shut down the session.")
